### Gold — Dimensions

Construit les tables de dimensions du modèle en étoile à partir des données silver déjà nettoyées. Chaque dimension reçoit une clé de substitution (surrogate key) : un entier technique, stable, indépendant de la clé naturelle source.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
df_ventes = spark.table("silver.ventes.ventes_clean")
df_retours = spark.table("silver.ventes.retours_clean")

In [0]:
# MAGIC Les valeurs distinctes de produit/catégorie viennent des ventes (les
# MAGIC retours ne référencent que `id_vente`, pas directement un produit — on
# MAGIC le retrouvera via la jointure en fact_retours plus tard si besoin).
# MAGIC 
# MAGIC `row_number()` sur une fenêtre ordonnée par `id_produit` donne un entier
# MAGIC croissant stable — équivalent ici à un auto-increment SQL classique.

window_produit = Window.orderBy("id_produit")

dim_produit = (
    df_ventes
    .select("id_produit", "produit", "categorie")
    .distinct()
    .withColumn("sk_produit", F.row_number().over(window_produit))
    .select("sk_produit", "id_produit", "produit", "categorie")
)

display(dim_produit)

In [0]:
window_region = Window.orderBy("region")

dim_region = (
    df_ventes
    .select("region")
    .distinct()
    .withColumn("sk_region", F.row_number().over(window_region))
    .select("sk_region", "region")
)

display(dim_region)

In [0]:
window_client = Window.orderBy("id_client")

dim_client = (
    df_ventes
    .select("id_client")
    .distinct()
    .withColumn("sk_client", F.row_number().over(window_client))
    .select("sk_client", "id_client")
)

display(dim_client)

In [0]:
window_vendeur = Window.orderBy("id_vendeur")

dim_vendeur = (
    df_ventes
    .select("id_vendeur")
    .distinct()
    .withColumn("sk_vendeur", F.row_number().over(window_vendeur))
    .select("sk_vendeur", "id_vendeur")
)

display(dim_vendeur)

In [0]:
# MAGIC Contrairement aux autres dimensions, celle-ci n'est **pas extraite** des
# MAGIC données — elle est **générée** de façon exhaustive sur toute la plage de
# MAGIC dates du projet (2022 à 2026), même les jours sans aucune vente. C'est
# MAGIC la pratique standard : une dimension temps incomplète empêcherait Power BI
# MAGIC de faire des comparaisons "mois vs mois précédent" correctement si un mois
# MAGIC entier n'a aucune transaction.
# MAGIC 
# MAGIC `sequence()` génère un tableau de dates, `explode()` transforme ce tableau
# MAGIC en une ligne par date.

# COMMAND ----------

dim_temps = (
    spark.sql("""
        SELECT explode(sequence(
            to_date('2022-01-01'),
            to_date('2026-12-31'),
            interval 1 day
        )) AS date_vente
    """)
    .withColumn("sk_temps", F.date_format("date_vente", "yyyyMMdd").cast("int"))
    .withColumn("annee", F.year("date_vente"))
    .withColumn("trimestre", F.quarter("date_vente"))
    .withColumn("mois", F.month("date_vente"))
    .withColumn("nom_mois", F.date_format("date_vente", "MMMM"))
    .withColumn("jour", F.dayofmonth("date_vente"))
    .withColumn("jour_semaine", F.date_format("date_vente", "EEEE"))
    .select("sk_temps", "date_vente", "annee", "trimestre", "mois", "nom_mois", "jour", "jour_semaine")
)

display(dim_temps.limit(10))

In [0]:
tables_dimensions = {
    "dim_produit": dim_produit,
    "dim_region": dim_region,
    "dim_client": dim_client,
    "dim_vendeur": dim_vendeur,
    "dim_temps": dim_temps,
}

for nom_table, df in tables_dimensions.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"gold.ventes.{nom_table}")
    )
    print(f"gold.ventes.{nom_table} : {df.count()} lignes")